In [3]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 135.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3

In [4]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import os
import pandas as pd
import re

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Loading a base model

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### Adding LoRA adapters

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

Unsloth 2026.8.22 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


### Loading the dataset

In [16]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rambo011/bhagavad-gita-q-and-a-dataset-for-modern-life-problem")

print("Path to dataset files:", path)

100%|██████████| 2.12M/2.12M [00:00<00:00, 146MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/rambo011/bhagavad-gita-q-and-a-dataset-for-modern-life-problem/versions/1


In [17]:
for f in os.listdir(path):

    print(f)

Chapter_18_QA.csv
Chapter_5_QA.csv
Chapter_13_QA.csv
Chapter_10_QA.csv
Chapter_14_QA.csv
Chapter_15_QA.csv
Chapter_16_QA.csv
Chapter_11_QA.csv
Chapter_3_QA.csv
Chapter_7_QA.csv
Chapter_8_QA.csv
Chapter_1_QA.csv
Chapter_4_QA.csv
Chapter_9_QA.csv
Chapter_2_QA.csv
Chapter_17_QA.csv
Chapter_12_QA.csv
Chapter_6_QA.csv


In [24]:
files = [
    "Chapter_18_QA.csv",
"Chapter_5_QA.csv",
"Chapter_13_QA.csv",
"Chapter_10_QA.csv",
"Chapter_14_QA.csv",
"Chapter_15_QA.csv",
"Chapter_16_QA.csv",
"Chapter_11_QA.csv",
"Chapter_3_QA.csv",
"Chapter_7_QA.csv",
"Chapter_8_QA.csv",
"Chapter_1_QA.csv",
"Chapter_4_QA.csv",
"Chapter_9_QA.csv",
"Chapter_2_QA.csv",
"Chapter_17_QA.csv",
"Chapter_12_QA.csv",
"Chapter_6_QA.csv",

]
data = pd.concat(
    [pd.read_csv(os.path.join(path, f)) for f in files],
    ignore_index=True
)


len(data)

12902

In [29]:
data = data[['question','answer']]
data

,question,answer
0,"MokshaPath, I feel so much pressure to constan...","My dear one, your weariness comes not from the..."
1,"I'm in a relationship where I give so much, bu...","Beloved seeker, your heart's pain arises from ..."
2,"I'm a parent, and I constantly worry about my ...","You worry, not because of your children's path..."
3,"I'm facing a huge decision about a job change,...","The demon of doubt, my child, holds you captiv..."
4,I feel so much anger and resentment towards so...,Your anger is a chain forged from your attachm...
...,...,...
12897,I procrastinate constantly. I know what I need...,Begin by offering your intention and your effo...
12898,I'm concerned about the state of the world – e...,"Dear one, your contribution, born of a heart a..."
12899,"I sometimes feel a deep sense of unworthiness,...","My child, your true worth is not measured by w..."
12900,I'm in a period of significant change – moving...,"In times of flux, anchor your inner being in t..."


In [30]:
dataset = data

### Train

In [ ]:
trainer = SFTTrainer(model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    formatting_func = format_prompt, # Add this line
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,        # small run for testing
        learning_rate = 2e-4,
        output_dir = "outputs",
        logging_steps = 1,
    ),
)

trainer.train()

Unsloth: not enough free memory for dataset tokenization workers (~1GB each); tokenizing in-process.


Unsloth: Tokenizing ["text"]:   0%|          | 0/51760 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51,760 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,1.488919
2,1.770598
3,1.821837
4,1.710012
5,1.287591
6,1.315940
7,1.376198
8,1.478132
9,1.236818
10,1.402924


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=60, training_loss=1.289132046699524, metrics={'train_runtime': 107.7733, 'train_samples_per_second': 4.454, 'train_steps_per_second': 0.557, 'total_flos': 840751107059712.0, 'train_loss': 1.289132046699524, 'epoch': 0.00927357032457496})

### Testing Fine Tuned Model

In [ ]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
       

In [ ]:
inputs = tokenizer(["### Instruction:\nExplain gravity simply\n\n### Response:\n"], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0]))

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain gravity simply

### Response:
Gravity is the force that attracts two objects with mass. It causes things to fall down and makes apples fall from trees, stars orbit around a planet, and planets move in elliptical orbits around the sun. The strength of gravity depends on how much mass an object has - the more massive something is, the stronger its gravitational pull will be. Gravity also helps keep us grounded by pulling our feet towards the ground when we stand or walk. Without gravity, there would be no weight, and everything would float


### Comparing to Instruct model

In [ ]:
instruct_model, instruct_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
inputs = instruct_tokenizer(["### Instruction:\nExplain gravity simply\n\n### Response:\n"], return_tensors="pt").to("cuda")
outputs = instruct_model.generate(**inputs, max_new_tokens=100)
print(instruct_tokenizer.decode(outputs[0]))

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain gravity simply

### Response:
Gravity is a force that attracts two objects with mass. It pulls things towards the center of Earth, keeping us on the ground and holding our feet to the floor. Gravity also causes objects to fall down if they are dropped or thrown upwards, and it keeps planets in orbit around stars. Gravity is an important concept in physics because it explains how everything in the universe interacts with each other. Without gravity, there would be no weight, no height, and no distance. Understanding gravity is essential for many fields
